In [ ]:
# Params
#=========================================================
# KNMI data
param_stationID = '260'
param_from_date = '1960'
param_to_date = '2000' 
param_vars = "average_temperature, minimum_temperature, maximum_temperature, total_precipitation, average_windspeed, average_winddirection, relative_humidity, minimum_humidity, maximum_humidity"

param_stationID_spinup = '260'
param_from_date_spinup = '1901' #Earliest possible date for this station is 1901
param_to_date_spinup = '1960' 
param_vars_spinup = "average_temperature, minimum_temperature, maximum_temperature, total_precipitation, average_windspeed, average_winddirection, relative_humidity, minimum_humidity, maximum_humidity"

# Loobos species
param_use_loobos_subset = 'yes' 

# Ecoregion map 
param_ecoregion_name = '101'

# Climate generator 
param_climate_time_series = 'Monthly_SequencedYears'
param_spinup_climate_time_series = 'Monthly_SequencedYears'

# Biomass initialization file 
param_timestep = '10'

# Scenario file 
param_duration = '100'
param_cell_length = '100'

In [ ]:
# Install and load all packages
#=========================================================

if (!require("pacman")) install.packages("pacman")
pacman::p_load("dplyr", "tidyr", "stringr", "lubridate", "readr", "Hmisc", "terra", "sf", "raster", "processx")


In [ ]:
# KNMI data retriever
#=========================================================
# Create directory
if(!dir.exists("/tmp/data")) dir.create("/tmp/data")
#=========================================================
# Part 1: Set up for KNMI daily data 
#=========================================================
# Parameter validation - check whether all parameters were filled in by user
if (param_stationID    == "" ||
    param_from_date    == "" ||
    param_to_date      == "" ||
    length(param_vars) == 0) {
  stop("All parameters must be provided by the user.")
}

# Split climate variables to make columns 
param_vars <- unlist(strsplit(param_vars, ",\\s*"))

param_vars[param_vars == "average_windspeed"]     <- "FG"
param_vars[param_vars == "average_temperature"]   <- "TG"
param_vars[param_vars == "minimum_temperature"]   <- "TN"
param_vars[param_vars == "maximum_temperature"]   <- "TX"
param_vars[param_vars == "total_precipitation"]   <- "RH"
param_vars[param_vars == "average_winddirection"] <- "DDVEC"
param_vars[param_vars == "relative_humidity"]     <- "UG"
param_vars[param_vars == "minimum_humidity"]      <- "UN"
param_vars[param_vars == "maximum_humidity"]      <- "UX"

#=========================================================
# Part2: Set up for KNMI spinup data 
#=========================================================
# Parameter validation - check whether all parameters were filled in by user
if (param_stationID_spinup    == "" ||
    param_from_date_spinup    == "" ||
    param_to_date_spinup      == "" ||
    length(param_vars_spinup) == 0) {
  stop("All parameters must be provided by the user.")
}

#Split climate variables to make columns and rename so that function can use set parameter values 
param_vars_spinup <- unlist(strsplit(param_vars_spinup, ",\\s*"))

param_vars_spinup[param_vars_spinup == "average_windspeed"]     <- "FG"
param_vars_spinup[param_vars_spinup == "average_temperature"]   <- "TG"
param_vars_spinup[param_vars_spinup == "minimum_temperature"]   <- "TN"
param_vars_spinup[param_vars_spinup == "maximum_temperature"]   <- "TX"
param_vars_spinup[param_vars_spinup == "total_precipitation"]   <- "RH"
param_vars_spinup[param_vars_spinup == "average_winddirection"] <- "DDVEC"
param_vars_spinup[param_vars_spinup == "relative_humidity"]     <- "UG"
param_vars_spinup[param_vars_spinup == "minimum_humidity"]      <- "UN"
param_vars_spinup[param_vars_spinup == "maximum_humidity"]      <- "UX"

#=========================================================
# Part3: Write function to get the KNMI data, this includes checks whether the dates are filled in correctly and logically 
# Does not require API because it is historic weather data 
#=========================================================
get_data <- function(stationID, from, to, vars) {
  
  # Date parsing - checking whether date is logical 
  if (!is.character(from) ||
      !is.character(to) ||
      stringr::str_length(from) %% 2 == 1 ||
      stringr::str_length(to) %% 2 == 1) {
    stop(
      "The values for 'from' and 'to' must be strings in the format 'YYYY', 'YYYYMM' or 'YYYYMMDD'."
    )
  }
  
  if (stringr::str_length(from) == 6) {
    from_date <- paste0(from, "01")
  } else if (stringr::str_length(from) == 4) {
    from_date <- paste0(from, "0101")
  } else {
    from_date <- from
  }
  
  if (stringr::str_length(to) == 8) {
    to_date <- to
  } else {
    if (stringr::str_length(to) == 6) {
      to <- paste0(to, "01")
    } else if (stringr::str_length(to) == 4) {
      to <- paste0(to, "1231")
    }
    
    to_date <- lubridate::ymd(to) |>
      lubridate::ceiling_date(unit = "month") - 1
    
    to_date <- gsub("-", "", as.character(to_date))
  }
  
  if (as.numeric(to_date) < as.numeric(from_date)) {
    stop("'from' must be earlier than or equal to 'to'")
  }
  
  # Station validation - check whether station can be selected. The list contains all the KNMI weather stations that collect daily data 
  selectable_stations <- c("260", "275", "209", "210", "215", "225", "229", "235", "240", "242", "248", "249", "251", "257", "258", "260", "265", 
                           "267", "269", "270", "273", "275", "277", "278", "279", "280", "283", "285", "286", "290", "308", "310", "311", "312", 
                           "313", "315", "316", "319", "323", "324", "330", "331", "340", "343", "344", "348", "350", "356", "370", "375", "377",
                           "380", "391", "392")  
  stationID <- as.character(stationID)
  
  if (!stationID %in% selectable_stations) {
    stop(
      paste0(
        "Invalid stationID. Choose one of: ",
        paste(selectable_stations, collapse = ", ")
      ))
  }
  
  # Build URL to retrieve the KNMI data from the website 
  baseURL <- "https://www.daggegevens.knmi.nl/klimatologie/daggegevens"
  URL <- paste0(
    baseURL,
    "?start=", from_date,
    "&end=", to_date,
    "&stns=", stationID,
    "&ALL"
  )
  
  # Download data from KNMI 
  data_daily <- read_csv(
    URL,
    col_names = FALSE,
    comment = "#",
    show_col_types = FALSE
  ) |>
    dplyr::as_tibble()
  
    colnames(data_daily) <-
      c(
        "STN", "YYYYMMDD", "DDVEC", "FHVEC", "FG", "FHX", "FHXH", "FHN", "FHNH", "FXX", "FXXH", "TG", "TN", "TNH", "TX", "TXH", "T10N",
        "T10NH", "SQ", "SP", "Q", "DR", "RH", "RHX", "RHXH", "PG", "PX", "PXH", "PN", "PNH", "VVN", "VVNH", "VVX", "VVXH", "NG",
        "UG", "UX", "UXH", "UN", "UNH", "EV24"
      )
    
  # Select only requested variables as indicated when parameters were set 
  data_daily <- data_daily |>
    #tidyr::drop_na(dplyr::all_of(vars)) |>
    dplyr::select(all_of(c("YYYYMMDD", vars)))
  
  # Scaling because KNMI works in 0.1
  if ("FG" %in% vars) data_daily$FG <- data_daily$FG / 10
  if ("TG" %in% vars) data_daily$TG <- data_daily$TG / 10
  if ("TN" %in% vars) data_daily$TN <- data_daily$TN / 10
  if ("TX" %in% vars) data_daily$TX <- data_daily$TX / 10
  if ("RH" %in% vars) data_daily$RH <- data_daily$RH / 10
  
  return(data_daily)
}

#=========================================================
# Part 4: Run the functions twice for both daily and spinup 
#=========================================================
KNMI_daily <- get_data(
    stationID = param_stationID,
    from      = param_from_date,
    to        = param_to_date,
    vars      = param_vars
)

KNMI_spinup <- get_data(
  stationID = param_stationID_spinup,
  from      = param_from_date_spinup,
  to        = param_to_date_spinup,
  vars      = param_vars_spinup
)

# Save the output as csv to tmp data to pass onto next cells 
KNMI_daily_datafile <- "/tmp/data/KNMI_daily.csv"
write.csv(KNMI_daily, 
          file = KNMI_daily_datafile, 
          row.names = FALSE)

KNMI_spinup_datafile <- "/tmp/data/KNMI_spinup.csv"
write.csv(KNMI_spinup, 
          file = KNMI_spinup_datafile, 
          row.names = FALSE)

In [ ]:
# KNMI data preprocessor
#=========================================================
# Read the file that was retrieved in previous cell 
KNMI_daily  <- read.csv(KNMI_daily_datafile)
KNMI_spinup <- read.csv(KNMI_spinup_datafile)

# Part 1: Preprocess the KNMI daily data 
# Reordering the date from YYYYMMDD to Year, Month, and Day in separate columns
Preprocessed_KNMI_daily <- KNMI_daily |>
  dplyr::mutate(YYYYMMDD = as.character(YYYYMMDD)) |>
  dplyr::mutate(
    date  = ymd(YYYYMMDD),
    Year  = year(date),
    Month = month(date),
    Day   = day(date)
  ) |>
  dplyr::select(-YYYYMMDD, -date) |>

# Renaming KNMI variables to match LANDIS-II names & reorder and rename columns 
  dplyr::rename(any_of(c(
    windSpeed = "FG",
    temp      = "TG",
    mintemp   = "TN",
    maxtemp   = "TX",
    precip    = "RH",
    RH        = "UG",
    maxRH     = "UX",
    minRH     = "UN",
    windDirection = "DDVEC"
  ))) |>
tidyr::pivot_longer(
    cols = -c(Year, Month, Day),
    names_to = "Variable",
    values_to = param_ecoregion_name
  ) |>
  dplyr::select(Year, Month, Day, Variable, {{param_ecoregion_name}}) |>
  tidyr::drop_na({{param_ecoregion_name}})

# Part 2: Preprocess the KNMI spinup data 
# Reordering the date from YYYYMMDD to Year, Month, and Day 
Preprocessed_KNMI_spinup <- KNMI_spinup |>
 dplyr::mutate(YYYYMMDD = as.character(YYYYMMDD)) |>
  dplyr::mutate(
    date  = ymd(YYYYMMDD),
    Year  = year(date),
    Month = month(date),
    Day   = day(date)
  ) |>
  dplyr::select(-YYYYMMDD, -date) |>

# Renaming KNMI variables to match LANDIS-II names & order and rename columns 
  dplyr::rename(any_of(c(
    windSpeed = "FG",
    temp      = "TG",
    mintemp   = "TN",
    maxtemp   = "TX",
    precip    = "RH",
    RH        = "UG",
    maxRH     = "UX",
    minRH     = "UN",
    windDirection = "DDVEC"      
  ))) |>
tidyr::pivot_longer(
  cols = -c(Year, Month, Day),
  names_to = "Variable",
  values_to = param_ecoregion_name  
) |>
  dplyr::select(Year, Month, Day, Variable, {{param_ecoregion_name}}) |>
  tidyr::drop_na({{param_ecoregion_name}})

# Part 3: Aggregate daily KNMI data to monthly values
Preprocessed_KNMI_daily[[param_ecoregion_name]] <- 
  as.numeric(Preprocessed_KNMI_daily[[param_ecoregion_name]])

KNMI_monthly <- Preprocessed_KNMI_daily |>
  dplyr::group_by(Year, Month, Variable) |>
  dplyr::summarise(
    value = ifelse(
      first(Variable) == "precip",
      sum(.data[[param_ecoregion_name]], na.rm = TRUE),
      mean(.data[[param_ecoregion_name]], na.rm = TRUE)
    ),
    .groups = "drop"
  ) |>
  dplyr::rename(!!param_ecoregion_name := value)

# Also for the spinup data
Preprocessed_KNMI_spinup[[param_ecoregion_name]] <- 
  as.numeric(Preprocessed_KNMI_spinup[[param_ecoregion_name]])

KNMI_monthly_spinup <- Preprocessed_KNMI_spinup |>
  dplyr::group_by(Year, Month, Variable) |>
  dplyr::summarise(
    value = ifelse(
      first(Variable) == "precip",
      sum(.data[[param_ecoregion_name]], na.rm = TRUE),
      mean(.data[[param_ecoregion_name]], na.rm = TRUE)
    ),
    .groups = "drop"
  ) |>
  dplyr::rename(!!param_ecoregion_name := value)

# Save the output as csv to the cloud storage so it can be inspected by user after preprocessing 
#Preprocessed_KNMI_daily_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_daily.csv"
#Preprocessed_KNMI_spinup_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_spinup.csv"
KNMI_monthly_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_monthly.csv"
KNMI_monthly_spinup_datafile <- "/home/jovyan/inputfiles/Preprocessed_KNMI_spinup_monthly.csv"

#write.csv(Preprocessed_KNMI_daily, 
        #  file = Preprocessed_KNMI_daily_datafile, 
         # quote = FALSE,
          # row.names = FALSE)

write.csv(KNMI_monthly, 
          file = KNMI_monthly_datafile, 
          quote = FALSE,
          row.names = FALSE)

write.csv(KNMI_monthly_spinup, 
          file = KNMI_monthly_spinup_datafile, 
          quote = FALSE,
          row.names = FALSE)

#write.csv(Preprocessed_KNMI_spinup, 
 #         file = Preprocessed_KNMI_spinup_datafile, 
  #        quote = FALSE,
   #       row.names = FALSE)

In [ ]:
# Species trait data retriever & preprocessor 
#=========================================================
# Location data sources
input_path <- file.path("/home/jovyan/Cloud Storage/naa-vre-public/vl-veluwe-forest-model/")

# Define set of species relevant to Loobos
loobos_species_vec = c("pinussylvestris", "quercus", "betula", "pseudotsugamenziesii") #not a changeable parameter

#=========================================================
# Part 1: Retrieve species trait data
#=========================================================
# Load and prepare TRY database for biomass succession
TRY_biomass <- read.delim(paste0(input_path, "try-data.txt"), fileEncoding = "latin1", dec = ".", quote = "", stringsAsFactors = FALSE)
TRY_biomass <- TRY_biomass |>
  dplyr::select(-"X") |>
  dplyr::mutate(
      genus = stringr::word(AccSpeciesName, 1),
      species = stringr::word(AccSpeciesName, 2),
      SpeciesCode = paste0(tolower(genus), tolower(species)),

      SpeciesCode = dplyr::case_when(
          SpeciesCode %in% c("quercusrobur", "quercuspetraea") ~ "quercus",
          stringr::str_detect(SpeciesCode, "betula") ~ "betula",
      TRUE ~ SpeciesCode
    )
  )

#=========================================================
# Part 2: Check if user wants to use the Loobos Subset
#=========================================================

if (param_use_loobos_subset == 'yes') {
  message("Filtering for Loobos species: ", paste(loobos_species_vec, collapse = ", ")) 

  loobos_sp <- TRY_biomass |>
    dplyr::filter(SpeciesCode %in% loobos_species_vec) 
    
} else {
  loobos_sp <- TRY_biomass
  message("Using the full dataset.")
}
      
#=========================================================
# Part 3: Prepare data for LANDIS-II
#=========================================================
# Write function to easily min, mean or max a value for a tree trait in TRY    
summarise_by_species <- function(data, stat = c("min", "max", "mean"), name = "result") {
  stat <- match.arg(stat)  # Validate input
  name_sym <- dplyr::sym(name)    # Convert name to symbol for tidy evaluation
  
  data |>
    dplyr::group_by(SpeciesCode) |>
    dplyr::summarise(
      !!name_sym := dplyr::case_when(
        stat == "min"  ~ min(OrigValueStr, na.rm = TRUE),
        stat == "max"  ~ max(OrigValueStr, na.rm = TRUE),
        stat == "mean" ~ round(mean(OrigValueStr, na.rm = TRUE), 3)
      ),
      .groups = "drop"
    )
}

# Prepare TRY database - clean dataset to obtain usable values
loobos_sp_cleaned <- loobos_sp |>
  dplyr::mutate(
    OrigValueStr = dplyr::na_if(OrigValueStr, ""), #Remove empty strings
    is_date = stringr::str_detect(OrigValueStr, "\\d{4}-\\d{2}-\\d{2}"),  # Flag dates
    number_list = stringr::str_extract_all(OrigValueStr, "\\d+\\.?\\d*"),  # Extract numbers
    OrigValueStr = sapply(number_list, function(x) {  # Overwrite with cleaned value
      x_num <- suppressWarnings(as.numeric(x))
      if (length(x_num) == 0 || all(is.na(x_num))) return(NA_real_)
      else return(mean(x_num, na.rm = TRUE))
    }),
    OrigUnitStr = stringr::str_to_lower(OrigUnitStr)  # Normalize OrigUnitStr to lowercase for easier case when matching
  ) |>
  dplyr::filter(!is.na(OrigValueStr), !.data$is_date) #keep only rows with valid values
       
#=========================================================
# Part 3a: Core species data
#=========================================================
# Prepare TRY database for LANDIS parameter sexual maturity (traitID 155)
      
TRY_155 <- loobos_sp_cleaned |>
  dplyr::filter(TraitID == 155)

# Make a column with mean age of flowering of trees, TRY misses Pseumenz and Larikaem
df_flower_age <- summarise_by_species(TRY_155, "mean", "Sexual Maturity") |>
  dplyr::mutate(
    "Sexual Maturity" = as.integer(round(.data$`Sexual Maturity`))
  ) |>
  dplyr::add_row(
      SpeciesCode = "pseudotsugamenziesii",
      "Sexual Maturity" = 12
    ) |>
  dplyr::add_row(
      SpeciesCode = "larixkaempferi",
      "Sexual Maturity" = 10
      )

# Prepare TRY database for LANDIS parameter plant longevity (traitID 59)
TRY_59 <- loobos_sp_cleaned |>
  dplyr::filter(.data$TraitID == 59)

# Make a column with max age of a tree
df_plant_age <- summarise_by_species(TRY_59, "max", "Longevity")

# Prepare TRY database for LANDIS parameter dispersal distance (traitID 193)
TRY_193 <- loobos_sp_cleaned |>
  dplyr::filter(TraitID == 193)

# Make a column with effective dispersal distance
df_eff_seed_dist <- TRY_193 |> 
  dplyr::filter(!stringr::str_starts(OriglName, "Max")) |>
  summarise_by_species("mean", "Seed Dispersal Dist Effective") |>
  dplyr::add_row(
    SpeciesCode = "betula",
    "Seed Dispersal Dist Effective" = 50
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercus",
    "Seed Dispersal Dist Effective" = 100
  ) |>
  dplyr::add_row(
    SpeciesCode = "fagussylvatica",
    "Seed Dispersal Dist Effective" = 30
  ) |>
  dplyr::add_row(
    SpeciesCode = "piceaabies",
    "Seed Dispersal Dist Effective" = 45
  ) |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    "Seed Dispersal Dist Effective" = 500
  )

# Make a column with maximum dispersal distance, TRY misses Querrubr, Betuspec, Fagusylv, Piceabie, Querspec and Larikaem
df_max_seed_dist <- TRY_193 |> 
  dplyr::filter(stringr::str_starts(OriglName, "Max")) |> # Include names starting with "Max"
  dplyr::filter(!stringr::str_starts(OriglName, "Eff")) |> #Exclude names starting with "Effective"
  summarise_by_species("max", "Seed Dispersal Dist Maximum") |>
  dplyr::add_row(
    SpeciesCode = "betula",
    "Seed Dispersal Dist Maximum" = 1000
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercus",
    "Seed Dispersal Dist Maximum" = 1500
  ) |>
  dplyr::add_row(
    SpeciesCode = "fagussylvatica",
    "Seed Dispersal Dist Maximum" = 70
  ) |>
  dplyr::add_row(
    SpeciesCode = "piceaabies",
    "Seed Dispersal Dist Maximum" = 200
  ) |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    "Seed Dispersal Dist Maximum" = 1000
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercusrubra",
    "Seed Dispersal Dist Maximum" = 1500
  )

# Prepare TRY database for resprouting probability (Traitid 819)
TRY_819 <- loobos_sp |>
  dplyr::filter(TraitID == 819) |>
  dplyr::filter(!stringr::str_detect(OrigUnitStr, "%")) |>	
  dplyr::mutate(
      OrigValueStr = dplyr::na_if(OrigValueStr, ""),  # Remove empty strings
      OrigValueStr = dplyr::case_when(
        OrigValueStr %in% c("No", "no", "1") ~ "0.0", #0.0 for no chance for resprouting
        OrigValueStr %in% c("Yes", "yes", "0", "moderate") ~ "0.8", #0.8 for all species that are able to resprout
        TRUE ~ OrigValueStr
      ),
      OrigValueStr = as.numeric(OrigValueStr)  # Convert to numeric
  ) |>
  dplyr::filter(!is.na(OrigValueStr))  # Keep only rows with valid values

# Make a column with resprouting probability of trees
df_resprout_prob <- summarise_by_species(TRY_819, "min", "Vegetative Reprod Prob") 
# min because Larix kaempferi had an able to resprout data point, however that paper did not mention larix kaempferi
# pinus sylvestris had an entry that had able to resprout, however, in that database the entry was not able to resprout
# picea abies had an entry that had able to resprout, however, there was no source in the database, and other sources specified that picea abies was definitely not able to resprout

# Incorporate minimum resprouting age
df_min_resprout_age <- tibble::tibble(
  SpeciesCode = character(),
  "Sprout Age Min" = numeric()
)

df_min_resprout_age <- df_min_resprout_age |>
  dplyr::add_row(
    SpeciesCode = "pinussylvestris",
    "Sprout Age Min" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "pseudotsugamenziesii",
    "Sprout Age Min" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "betula",
    "Sprout Age Min" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercus",
    "Sprout Age Min" = 20
  ) |>
  dplyr::add_row(
    SpeciesCode = "fagussylvatica",
    "Sprout Age Min" = 20
  ) |>
  dplyr::add_row(
    SpeciesCode = "piceaabies",
    "Sprout Age Min" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    "Sprout Age Min" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercusrubra",
    "Sprout Age Min" = 20
  )

#Incorporate maximum resrprouting age
df_max_resprout_age <- tibble::tibble(
  SpeciesCode = character(), 
  "Sprout Age Max" = numeric()
)

df_max_resprout_age <- df_max_resprout_age |>
  dplyr::add_row(
    SpeciesCode = "pinussylvestris",
    "Sprout Age Max" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "pseudotsugamenziesii",
    "Sprout Age Max" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "betula",
    "Sprout Age Max" = 100
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercus",
    "Sprout Age Max" = 150
  ) |>
  dplyr::add_row(
    SpeciesCode = "fagussylvatica",
    "Sprout Age Max" = 150
  ) |>
  dplyr::add_row(
    SpeciesCode = "piceaabies",
    "Sprout Age Max" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    "Sprout Age Max" = 0
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercusrubra",
    "Sprout Age Max" = 200
  )

# Prepare TRY database for method of post fire regenaration (Traitid 318 and 819)
TRY_318.819 <- loobos_sp |>
  dplyr::filter(TraitID %in% c(318, 819)) |>
  dplyr::mutate(
    OrigValueStr = dplyr::case_when( #resprout and serotiny are in different TRY traitID, serotiny takes priority over resprout over none
      TraitID == 318 & OriglName == "SeedlEmerg" & .data$OrigValueStr %in% c("low", "yes") ~ "serotiny",
      TraitID == 819 & OrigValueStr %in% c("No", "no", "1") ~ "none", #TRY paper uses 1 for no
      TraitID == 819 & OrigValueStr %in% c("Yes", "yes", "0", "moderate", "70") ~ "resprout", #TRY paper uses 0 for yes
      TRUE ~ NA_character_
    )
  ) |>
  dplyr::filter(!is.na(OrigValueStr)) |>
  dplyr::group_by(SpeciesCode) |>
  dplyr::summarise(
    OrigValueStr = dplyr::case_when(
      any(OrigValueStr == "serotiny") ~ "serotiny",
      any(OrigValueStr == "none") ~ "none",
      any(OrigValueStr == "resprout") ~ "resprout"),
    .groups = "drop"
  )

df_postfire_regen <- TRY_318.819 #not with function, since value is not numerical

# Make final input file for core species data
core_species_data <- df_plant_age |>
  dplyr::full_join(df_flower_age, by = "SpeciesCode") |>
  dplyr::full_join(df_eff_seed_dist, by = "SpeciesCode") |>
  dplyr::full_join(df_max_seed_dist, by = "SpeciesCode") |>
  dplyr::full_join(df_resprout_prob, by = "SpeciesCode") |>
  dplyr::full_join(df_min_resprout_age, by = "SpeciesCode") |>
  dplyr::full_join(df_max_resprout_age, by = "SpeciesCode") |>
  dplyr::full_join(df_postfire_regen, by = "SpeciesCode") |>
  # Add dummy values for 'rest' category for now - need fixing later
  dplyr::add_row(
      SpeciesCode = "rest",
      "Longevity" = 500,
      "Sexual Maturity" = as.integer(median(df_flower_age$`Sexual Maturity`)),
      "Seed Dispersal Dist Effective" = 50,
      "Seed Dispersal Dist Maximum" = 1500,
      "Vegetative Reprod Prob" = 0,
      "Sprout Age Min" = 0,
      "Sprout Age Max" = 0,
      OrigValueStr = "none"
  )

# Filter for species relevant to Loobos
if (param_use_loobos_subset == 'yes') {
  core_species_data <- core_species_data |>
    dplyr::filter(SpeciesCode %in% loobos_species_vec)
}

core_species_data <- apply(core_species_data, 1, function(row) paste(row, collapse = "\t"))

header <- c(
  "LandisData Species"
)

# Save the output as a txt.file to the Cloud storage so it can be used in later cells and be inspected by the user 
output_lines <- c(header, core_species_data)
loobos_core_species_datafile <- "/home/jovyan/inputfiles/loobos_core_species_data.txt"
writeLines(output_lines, loobos_core_species_datafile)

#=========================================================
# Part 3b: SppEcoregionData for biomass succession
#=========================================================

# Make an if-loop
# Also read in establishment values from Probos. Probos misses Querrubr
df_prob_establish <- readr::read_csv(paste0(input_path, "establishmentTreeVeluwe.csv")) |>
  dplyr::select(-1) |>              # Remove first column if it's row names
  dplyr::summarise(dplyr::across(dplyr::everything(), ~sum(.x, na.rm = TRUE))) |>
  tidyr::pivot_longer(
    cols = everything(), 
    names_to = "SpeciesCode", 
    values_to = "ProbEstablish"
  ) |>
  dplyr::mutate(
    Year = 0,
    EcoregionName = param_ecoregion_name,
    ProbEstablish = pmin(ProbEstablish / 8, 1)
  )

df_prob_establish <- df_prob_establish |>
  dplyr::add_row(
    SpeciesCode = "quercusrubra",
    ProbEstablish = 0,
    Year = 0, 
    EcoregionName = param_ecoregion_name
  )

# Read in vitality scores from NBI-7
mortality <- dplyr::tibble(
  SpeciesCode = c("quercusrubra", "betula", "fagussylvatica", "pseudotsugamenziesii", "piceaabies", "pinussylvestris", "quercus", "larixkaempferi"),
  ProbMortality = c(0.0074, 0.0031, 0.0095, 0.0102, 0.0167, 0.0023, 0.0209, 0.0056)
)

# Make dataframe for Probmortality
df_prob_mortality <- mortality |>
  dplyr::mutate(
    EcoregionName = param_ecoregion_name,
    Year = 0
  )

# Incorporate ANPP_max for species - default values used 
df_anpp_max <- tibble::tibble(
  SpeciesCode = character(),
  "ANPPMax" = numeric()
)

df_anpp_max <- df_anpp_max |>
  dplyr::add_row(
    SpeciesCode = "pinussylvestris",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "pseudotsugamenziesii",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "betula",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercus",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "fagussylvatica",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "piceaabies",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    "ANPPMax"= 100000
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercusrubra",
    "ANPPMax"= 100000
  ) |>
  dplyr::mutate(
      EcoregionName = param_ecoregion_name,
      Year = 0
  )

# Incorporate Maximum Biomass (BiomassMax)
df_biomass_max <- tibble::tibble(
  SpeciesCode = character(),
  "BiomassMax" = numeric()
)

df_biomass_max <- df_biomass_max |>
  dplyr::add_row(
    SpeciesCode = "pinussylvestris",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "pseudotsugamenziesii",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "betula",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercus",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "fagussylvatica",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "piceaabies",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    "BiomassMax"= 10000000
  ) |>
  dplyr::add_row(
    SpeciesCode = "quercusrubra",
    "BiomassMax"= 10000000
  ) |>
  dplyr::mutate(
      EcoregionName = param_ecoregion_name,
      Year = 0
  )
     
# Make final file for Species Ecoregion Data
SppEcoregionData.biomass <- df_prob_establish |>
  dplyr::full_join(df_prob_mortality, by = c("SpeciesCode", "Year", "EcoregionName")) |>
  dplyr::full_join(df_anpp_max, by = c("SpeciesCode", "Year", "EcoregionName")) |>
  dplyr::full_join(df_biomass_max, by = c("SpeciesCode", "Year", "EcoregionName"))
  
# Add dummy values for 'rest' category for now - need fixing later
SppEcoregionData.biomass <- SppEcoregionData.biomass |>
  dplyr::add_row(
    SpeciesCode = "rest",
    ProbEstablish = 1,
    Year = 0,
    EcoregionName = "101",
    ProbMortality = mean(df_prob_mortality$ProbMortality),
    ANPPMax = mean(df_anpp_max$ANPPMax),
    BiomassMax = 10000000
  ) |> 
  dplyr::select("Year", "EcoregionName", "SpeciesCode", "ProbEstablish", "ProbMortality", "ANPPMax", "BiomassMax") 

# Filter for species relevant to Loobos
if (param_use_loobos_subset == 'yes') {
  SppEcoregionData.biomass <- SppEcoregionData.biomass |>
    dplyr::filter(SpeciesCode %in% loobos_species_vec)
}

# Save the Species Ecoregion Data file as csv to the Cloud storage so it can be used in later cells and inspected by user 
loobos_spp_ecoregion_datafile <- "/home/jovyan/inputfiles/loobos_spp_ecoregion_data.csv"
readr::write_csv(SppEcoregionData.biomass, loobos_spp_ecoregion_datafile)

#=========================================================
# Part 3c: SpeciesDataBiomass
#=========================================================
# Prepare TRY database for LANDIS parameter leaf longevity (traitID 12)
TRY_12 <- loobos_sp |>
  dplyr::filter(.data$TraitID == 12) |>
  dplyr::mutate(
    # Remove non-numeric characters from OrigValueStr
    OrigValueStr = stringr::str_extract(OrigValueStr, "\\d+\\.?\\d*"),
    OrigValueStr = as.numeric(OrigValueStr),
    # Normalize OrigUnitStr to lowercase for easier case when matching
    OrigUnitStr = stringr::str_to_lower(OrigUnitStr),
    # Apply unit conversion
    OrigValueStr = dplyr::case_when(
      OrigUnitStr == "days" ~ OrigValueStr / 365,
      OrigUnitStr %in% c("month", "months") ~ OrigValueStr / 12,
      OrigUnitStr %in% c("year", "yr-1", "year-1") ~ OrigValueStr,
      TRUE ~ NA_real_  # Handle unexpected units
    ),
    # Enforce minimum value of 1, because deciduous trees in landis should have a value of 1
    OrigValueStr = dplyr::if_else(OrigValueStr < 1, 1, OrigValueStr),
    # Standardize unit label
    OrigUnitStr = "year"
  ) |>
  dplyr::filter(!is.na(OrigValueStr))  

# Make a column with mean longevity of leafs, TRY misses Larikaem
df_leaf_longevity <- summarise_by_species(TRY_12, "mean", "LeafLongevity") |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    LeafLongevity = 1
  )

# Prepare TRY database for Landis parameter wood decay rate (traitID 1158)
TRY_1158 <- loobos_sp |>
  dplyr::filter(TraitID == 1158) |>
  dplyr::filter(OriglName == "single exponential model: Yt=Yoe-kt") |>
  dplyr::filter(OrigValueStr >= 0) |>
  dplyr::mutate(OrigValueStr = as.numeric(OrigValueStr))

# Make a column with mean of wood decay rate, k in Yt=Yoe-kt, TRY misses Larikaem
df_wood_decayrate <- summarise_by_species(TRY_1158, "mean", "WoodDecayRate") |>
  dplyr::add_row(
    SpeciesCode = "larixkaempferi",
    WoodDecayRate = 0.021
  )

# Make dataframe for mortalitycurve
df_mortality_curve <- dplyr::tibble(
  SpeciesCode = c("larixkaempferi", "pseudotsugamenziesii", "piceaabies", "pinussylvestris", "fagussylvatica", "quercus", "quercusrubra", "betula"),
  MortalityCurve = c(10, 10, 10, 10, 10, 10, 10, 10)
)
      

# Make dataframe for growthcurve
df_growth_curve <- dplyr::tibble(
  SpeciesCode = c("larixkaempferi", "pseudotsugamenziesii", "piceaabies", "pinussylvestris", "fagussylvatica", "quercus", "quercusrubra", "betula"),
  GrowthCurve = c(0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9)
)

# Prepare TRY database for LANDIS parameter leaf lignin content (traitID 87)
TRY_87 <- loobos_sp |>
  dplyr::filter(TraitID == 87) |>
  dplyr::mutate(
    # Remove non-numeric characters from OrigValueStr
    # Turn percentages into proportions
    OrigValueStr = as.numeric(OrigValueStr) / 100
  ) |>
  dplyr::filter(!is.na(OrigValueStr))  

# Make a column with mean of leaf lignin content, TRY misses Larikaem, Pseumenz, Piceabie, Fagusylv, Querspec, Querrubr
df_leaf_lignin <- summarise_by_species(TRY_87, "mean", "LeafLignin") |>
  dplyr::add_row(
    SpeciesCode = c("larixkaempferi", "pseudotsugamenziesii", "piceaabies", "fagussylvatica", "quercus", "quercusrubra"),
    LeafLignin = c(0.21, 0.19, 0.18, 0.15, 0.17, 0.15)
  )

# Prepare TRY database for Landis parameter shade tolerance (traitID 603)
TRY_603 <- loobos_sp |>
  dplyr::filter(TraitID == 603) |>
  # Remove rows with "%" in OrigUnitStr
  dplyr::filter(!stringr::str_detect(OrigUnitStr, "%")) |>
  # Normalize OrigValueStr to lowercase for easier case_when usage
  dplyr::mutate(OrigValueStr = stringr::str_to_lower(OrigValueStr)) |>
  # Map textual values to numeric scale
  dplyr::mutate(
    OrigValueStr = dplyr::case_when(
      OrigValueStr %in% c("intolerant") ~ 1,
      OrigValueStr %in% c("indifferent - intolerant") ~ 2,
      OrigValueStr %in% c("intermediate") ~ 3,
      OrigValueStr %in% c("indifferent") ~ 4,
      OrigValueStr %in% c("tolerant") ~ 5,
      OrigValueStr %in% as.character(1:9) ~ dplyr::case_when(
        OrigValueStr == "1" ~ 5,
        OrigValueStr == "2" ~ 4.5,
        OrigValueStr == "3" ~ 4,
        OrigValueStr == "4" ~ 3.5,
        OrigValueStr == "5" ~ 3,
        OrigValueStr == "6" ~ 2.5,
        OrigValueStr == "7" ~ 2,
        OrigValueStr == "8" ~ 1.5,
        OrigValueStr == "9" ~ 1
      ),
      TRUE ~ NA_real_  # Remove other non-numeric or unknown values
    ) 
) |>
  # Remove rows with NA values after transformation
  dplyr::filter(!is.na(OrigValueStr)) |>
  # Standardize OrigUnitStr label
  dplyr::mutate(OrigUnitStr = "tolerance_score")

# Make a column with mean of shade tolerance
df_shade_tolerance <- summarise_by_species(TRY_603, "mean", "ShadeTolerance") |>
  dplyr::mutate(ShadeTolerance = round(ShadeTolerance)) #Landis requires integer input in this category

# Prepare TRY database for Landis parameter fire tolerance (traitID 318)
TRY_318 <- loobos_sp |>
  dplyr::filter(TraitID == 318) |>
  dplyr::filter(OriglName != "SeedlEmerg") |>
  # Step 2: Normalize OrigUnitStr to lowercase
  dplyr::mutate(OrigValueStr = stringr::str_to_lower(OrigValueStr)) |>
  # Map textual values to numeric scale
  dplyr::mutate(
    OrigValueStr = dplyr::case_when(
      OrigValueStr %in% c("no", "none") ~ 1,
      OrigValueStr %in% c("low", "surface fire") ~ 2,
      OrigValueStr %in% c("yes") ~ 4,
      TRUE ~ NA_real_  # Remove other non-numeric or unknown values
    )
  ) |>                         
  # Remove rows with NA values after transformation
  dplyr::filter(!is.na(OrigValueStr)) |>
  # Standardize OrigUnitStr label
  dplyr::mutate(OrigUnitStr = "tolerance_score")

# Make a column with mean of fire tolerance, TRY misses Querspec and Fausylv
df_fire_tolerance <- summarise_by_species(TRY_318, "mean", "FireTolerance") |>
  dplyr::mutate(FireTolerance= round(FireTolerance)) |> #Landis requires integer input in this category 
  dplyr::add_row(
    SpeciesCode = c("quercus", "fagussylvatica"),
    FireTolerance = c(3, 2)
  )

# Make final file for SpeciesData
SpeciesData.biomass <- df_leaf_longevity |>
  dplyr::full_join(df_wood_decayrate, by = "SpeciesCode") |>
  dplyr::full_join(df_mortality_curve, by = "SpeciesCode") |>
  dplyr::full_join(df_growth_curve, by = "SpeciesCode") |>
  dplyr::full_join(df_leaf_lignin, by = "SpeciesCode") |>
  dplyr::full_join(df_shade_tolerance, by = "SpeciesCode") |>
  dplyr::full_join(df_fire_tolerance, by = "SpeciesCode")

SpeciesData.biomass <- SpeciesData.biomass |>
  # Add dummy values for 'rest' category for now - need fixing later
  dplyr::add_row(
      SpeciesCode = "rest",
      LeafLongevity = 1,
      WoodDecayRate = mean(df_wood_decayrate$WoodDecayRate),
      MortalityCurve = 5,
      GrowthCurve = 0.9,
      LeafLignin = mean(df_leaf_lignin$LeafLignin),
      ShadeTolerance = 3,
      FireTolerance = 1
  )

# Filter for species relevant to Loobos
if (param_use_loobos_subset == 'yes') {
  SpeciesData.biomass <- SpeciesData.biomass |>
    dplyr::filter(SpeciesCode %in% loobos_species_vec)
}

# Save the Species Data file as csv to the Cloud storage so it can be used in later cells and inspected by user 
loobos_species_datafile <- "/home/jovyan/inputfiles/loobos_species_data.csv"
readr::write_csv(SpeciesData.biomass, loobos_species_datafile)

In [ ]:
# Loobos initial community generator
#=========================================================
# Utilise 1960 as starting date
init_comm <- tibble::tibble(
  SpeciesCode = character(),
  CohortAge = numeric(), 
  MapCode = numeric(),
  dbh_cm = numeric()
)

init_comm <- init_comm |>
  dplyr::add_row(
    SpeciesCode = "pinussylvestris",
    CohortAge = 10,
    MapCode = 1,
    dbh_cm = 6
  )
# Calculate biomass

# Determine volume based on formula and constants obtained from NBI7: https://doi.org/10.18174/571720
b0 <- 13.79
b1 <- -5.92
b2 <- 0.85

init_comm_loo <- init_comm |>
  dplyr::mutate(
    "volume_dm3" = b0 + b1 * dbh_cm + b2 * dbh_cm^2, #assume dbh is 6 for all trees
    "volume_m3" = volume_dm3 / 1000
  )

 #Biomass formula: e^(log(c0) + c1 * log(dbh_cm)) * cf; values from Forrester et al. 2017 https://doi.org/10.1016/j.foreco.2017.04.011

init_comm_loo <- init_comm_loo |>
  dplyr::mutate(
    "stem_biomass_kg" = volume_m3 * 420 #wood density in ton DM/m3 converted to kg/m3
  ) |>
  dplyr::mutate(
    "branch_biomass_kg" = exp(1)^(-3.6641 + 2.1601 * log(dbh_cm)) * 1.0451
  ) |>
  dplyr::mutate(
    "leaf_biomass_kg" =  exp(1)^(-3.5276 + 1.7471 * log(dbh_cm)) * 1.0104
  )

# Calculate total aboveground biomass
init_comm_loo <- init_comm_loo |>
  dplyr::mutate(
    "total_abg_biomass_kg" = stem_biomass_kg + branch_biomass_kg + leaf_biomass_kg,
  )

# Convert biomass into g/m2 that is valuable for Landis-ii
init_comm_loo <- init_comm_loo |>
  dplyr::mutate(
    "biomass_kg_ha" = total_abg_biomass_kg * 350 #trees per hectare obtained from Michiel
  ) |>
  dplyr::mutate(
    "biomass_g_m2" = (biomass_kg_ha*1000) / 10000 #kg/ha -> g/ha -> g/m2
  )


#===============================================================
# Create initial community file
#================================================================

loobos_initial_communities <- init_comm_loo |>
  dplyr::rename(
    SpeciesName = SpeciesCode,
    CohortBiomass =  biomass_g_m2
  ) |>
  dplyr::select(
    MapCode, SpeciesName, CohortAge, CohortBiomass
  )

loobos_initial_communities_datafile <- "/home/jovyan/inputfiles/initial_communities.csv"
readr::write_csv(loobos_initial_communities, loobos_initial_communities_datafile)  

#===============================================================
# Create initial community map
#================================================================
loobos_sf_datafile <- "/home/jovyan/inputfiles/loobos_digit.shp"

# Read in Loobos shapefile - should upload to cloud storage or LTER-LIFE data platform. 
loobos_sf <- sf::read_sf(loobos_sf_datafile)

# Transform to spatial vector
loobos_vect <- terra::vect(loobos_sf)

# Create Raster
raster_loobos <- terra::rast(
  loobos_vect,
  resolution = 100, #resolution could be changed for smaller grid cells
  crs = "EPSG:28992" # RD coordinate system
)

init_comm_raster <- terra::rasterize(
  loobos_vect,
  raster_loobos,
  field = 1, #assigns value 1 to all cells
  background = 0
)

loobos_initial_communities_tif <- "/home/jovyan/inputfiles/loobos_initial_communities.tif"
terra::writeRaster(
  init_comm_raster, 
  filename = loobos_initial_communities_tif, 
  datatype = "INT1U",
  overwrite = TRUE
)


In [ ]:
# EcoregionsMap creator 
#=========================================================
# Read loobos geopackage that was created in previous cell 
initial_communities_raster <- raster(loobos_initial_communities_tif)

# The ecoregion name is set to 101 because this matches with the example LANDIS files 
# This ecoregion corresponds to value 1; empty cells are set to 0
terra::values(initial_communities_raster) <- dplyr::if_else(is.na(terra::values(initial_communities_raster)), 0, 1)

# This file is saved to tmp data so that it can be used in later cells 
loobos_ecoregions_tif <- "/home/jovyan/inputfiles/loobos_ecoregions.tif"
writeRaster(
  initial_communities_raster, 
  filename = loobos_ecoregions_tif, 
  datatype = "INT1U",
  overwrite = TRUE
)

In [ ]:
# Configuration files creator  
#=========================================================
# Create path for the ecoregions .txt to be saved to 
loobos_ecoregions_datafile <- "/home/jovyan/inputfiles/loobos_ecoregions.txt"

# Create the ecoregions .txt file
# MapCode 0 corresponds to empty cells, which are set to inactive
# MapCode 1 corresponds to our single ecoregion, which is active
writeLines(c('LandisData  "Ecoregions"',
"",
">>         	     Map",
">> 	Active       Code  	      Name   	Description",
">> 	------       ---- 	     -----  	-----------",
paste('no', 0, '0', 'empty-cell', sep = '\t'),
paste('yes', 1, param_ecoregion_name, 'managed', sep = '\t')),
           loobos_ecoregions_datafile)
  
# Create path for the climategenerator .txt file to be saved to 
climategenerator_datafile <- "/home/jovyan/inputfiles/climategenerator.txt"

#Create the climategenerator .txt file
writeLines(c('LandisData "Climate Config"',
            paste('ClimateTimeSeries', param_climate_time_series, sep = "\t"),
            paste('ClimateFile', KNMI_monthly_datafile, sep = "\t"),
            paste('SpinUpClimateTimeSeries', param_spinup_climate_time_series, sep = "\t"),
            paste('SpinUpClimateFile', KNMI_monthly_spinup_datafile, sep = "\t"),
            '',
            paste('GenerateClimateOutputFiles', 'yes', sep = "\t"), 
            paste('UsingFireClimate', 'no  << Optional parameter; default is no.', sep = "\t"),
            paste('>>FineFuelMoistureCode', '100', sep = "\t"),
            paste('>>DuffMoistureCode', '100', sep = "\t"),
            paste('>>DroughtCode', '100', sep = "\t"),
            paste('>>FirstDayFire', '30', sep = "\t"),
            paste('>>LastDayFire', '320', sep = "\t")), 
              climategenerator_datafile)

In [ ]:
#Biomass Succession initialization file creator 
#=========================================================
# Create path for the Biomass Succession .txt file to be saved to 
loobos_biomass_succession_datafile <- "/home/jovyan/inputfiles/loobos_biomass_succession.txt" 

# Create the Biomass Succession .txt file 
writeLines(c('LandisData  "Biomass Succession"', 
'',
'',
'>>------------------',
'>> REQUIRED INPUTS',
'>>------------------',
'',
paste('Timestep', param_timestep, sep = "\t"),
'',
paste('SeedingAlgorithm', 'WardSeedDispersal', sep = "\t"), 
'',
paste('InitialCommunities', loobos_initial_communities_datafile, sep = "\t"),
paste('InitialCommunitiesMap', loobos_initial_communities_tif, sep = "\t"),
paste('ClimateConfigFile', climategenerator_datafile, sep = "\t"),
'',
paste('CalibrateMode', 'no << optional parameter', sep = "\t"),
'',
'',
'>>----------------------------',
'>> LIFE HISTORY PARAMETERS',
'>>----------------------------',
'',
'MinRelativeBiomass',
'>> Shade	Percent Max Biomass',
'>> Class	by Ecoregions',
'>> ----------	              --------------------',	
    paste('', param_ecoregion_name, sep = "\t"),
	paste('1', '25%', sep = "\t"),
	paste('2', '45%', sep = "\t"), 
	paste('3', '56%', sep = "\t"), 
	paste('4', '70%', sep = "\t"), 
	paste('5', '90%', sep = "\t"),
'',
'',
'SufficientLight',
paste('>> Spp Shade', 'Probability', sep = "\t"),
'>> Class	by Actual Shade',
'>> ----------	--------------------',	
'>> 		0 	     1 	     2 	     3       4 	     5',
   '1 		1.0 	0.5 	0.25 	0.0 	0.0 	0.0',
   '2 		1.0 	1.0 	0.5 	0.25 	0.0 	0.0',
   '3 		1.0 	1.0 	1.0 	0.5 	0.25 	0.0',
   '4 		1.0 	1.0 	1.0 	1.0 	0.5 	0.25',
   '5 		0.0 	0.0 	1.0 	1.0 	1.0 	1.0',
'',
'',
paste('SpeciesDataFile', loobos_species_datafile, sep = "\t"), 
'',
'EcoregionParameters',
'>>	AET (mm)',
paste(param_ecoregion_name, '500', sep = "\t"),
'',
paste('SpeciesEcoregionDataFile',  loobos_spp_ecoregion_datafile, sep = "\t"), 
'',
'FireReductionParameters',
'>>	Severity	WoodLitter	Litter',	
'>>	Fire		Reduct		Reduct',	
	'1		      0.0		0.5',
	'2		      0.0		0.75',	
	'3		      0.0		1.0',
'',
'HarvestReductionParameters',
'>> Name \t WoodLitter \t Litter \t Cohort \t Cohort',
'>>      \t Reduct \t Reduct \t WoodRemoval \t LeafRemoval',
paste('MaxAgeClearcut', 0.5, 0.15, 0.8, '0.0', sep = "\t"),
paste('PatchCutting', 1.0, 1.0, 1.0, '0.0', sep = "\t")), 
loobos_biomass_succession_datafile)

In [ ]:
# Scenario file creator 
#=========================================================
# Create path for the Scenario .txt file to be saved to 
loobos_scenario_datafile <- "/home/jovyan/inputfiles/loobos_scenario.txt"

# Create the Scenario .txt file 
writeLines(c('LandisData  "Scenario"',
'', 
'',
'',
'>>-------------------',
'>> REQUIRED INPUTS',
'>>-------------------',
'',
paste('Duration', param_duration, sep = "\t"),
'',
paste('Species', loobos_core_species_datafile, sep = "\t"),
'',
paste('Ecoregions', loobos_ecoregions_datafile, sep = "\t"),
paste('EcoregionsMap', loobos_ecoregions_tif, sep = "\t"),
'',
paste('CellLength', param_cell_length, sep = "\t"),
'',
'',
'',
'>> -----------------------',
'>> SUCCESSION EXTENSIONS',
'>> -----------------------',
'',
'>> 	Succession Extension     Initialization File',
'>> 	--------------------     -------------------',
paste('"Biomass Succession"', loobos_biomass_succession_datafile, sep = "\t"),
'',
'',
'>> --------------------------',
'>> DISTURBANCE EXTENSIONS',
'>> -------------------------',
'',
'>> 	Disturbance Extension	Initialization File',
'>>	    ---------------------	-------------------',
'',
'',
'>>   DisturbancesRandomOrder  no  	<< optional << Commented (default) is no',
'',                         	
'',
'>> ------------------------',
'>> OUTPUT EXTENSIONS',
'>> ----------------------',
'',
'>> 	Output Extension		Initialization File',
'>> 	----------------		-------------------',
'',
'',
'',
'>> RandomNumberSeed  147        << optional parameter; uncomment for reproducibilty tests',
'                                << Commented (default) is a RandomNumberSeed generated using the current time'), 
           loobos_scenario_datafile)

In [ ]:
# LANDIS II runner
#=========================================================
# Get Biomass Succession code from GitHub
if(!dir.exists("Extension-Biomass-Succession")) system("git clone https://github.com/LANDIS-II-Foundation/Extension-Biomass-Succession.git")

# Set working directory to Biomass Succession folder
#setwd("Extension-Biomass-Succession/testings/CoreV8.0-BiomassSuccession7.0")

# Create function to track the progress of LANDIS-II in notebooks
system_with_progress <- function(cmd) {
  cb <- function(line, proc) {
    print(line)
    flush.console()
  }
  cmd <- strsplit(cmd, " ")[[1]]
  res <- processx::run(cmd[1], cmd[-1], stdout_line_callback=cb, stderr_line_callback=cb)
  return(res)
}

# Run LANDIS-II
res <- system_with_progress(paste("dotnet", Sys.getenv("LANDIS_CONSOLE"), loobos_scenario_datafile))